# Session 02 Sandbox · The Q3 Expenditure Compliance Review

**ARI1101 Foundations of Data Science** · Data formats: CSV, JSON, XML and documents

## The question

Brightwater Logistics Ltd, a fictional company, has asked its data scientist a question:

> **How many payments above €5,000 were made in the last quarter (Q3 2026), and were they in line with the company's Expenditure Approval Policy?**

The answer is needed in a short report to the Board, with text, tables and a chart.

Answering it needs evidence from seven sources, in seven different formats: the policy (a Word document), the finance system's payment export (CSV), the bank's statement feed (JSON), the approval workflow export (XML), the approved vendor list (Excel), a set of emails (plain text) and the Board's own finance summary (PDF). The evidence has to be extracted, combined, checked and turned into a report.

## What you do and what the kit does

Doing all of that from scratch is not a first-session task. This notebook uses **ai-learning-kit**, a small library written for the AI study units, which hides the steps that are not today's topic. Today's topic is **data formats**, so that is the part you do yourself.

| You do (today's topic) | The kit does (hidden for now) |
|---|---|
| Inspect each source and identify its format and shape | Extract text, rules and tables from Word, Excel and PDF files |
| Read CSV, JSON and XML with `pandas`, `json` and `ElementTree` | Clean the data and convert currencies inside its own checks |
| Store extracted information in a suitable format (JSON or CSV) and read it back | Apply the policy rules to every transaction |
| Combine sources into one table and choose a format for it | Lay out the final report |
| Compare what each format keeps and loses | |

Later sessions use the same company and the same data. Each time, the kit hands a different step to you: cleaning and duplicates in the Scrub session, charts and statistics in Explore, and so on.

## Looking inside the black box

Nothing in the kit is secret. `kit.explain(function)` prints a plain-language description of any kit function, and `kit.source(function)` prints its Python code. Use them whenever you want to know how a step was done.

## Recording your answers

Each task ends with a call such as

```python
kit.answer("T3.1", value=..., reasoning="...")
```

which saves your answer and your reasoning in an **answer book**, a JSON file in the folder `answers/` next to this notebook (see Setup below). Where possible the kit gives immediate feedback: ✓ when the answer is right, • when your lecturer will read it, and ✗ when it is not yet right. A ✗ lists which parts of your answer are right and which are not, and for each wrong part what you recorded, the evidence in the case files that shows why, and a line of code you can run to check it yourself. For written answers the kit lists the points a good answer covers and marks those it finds (○ means "not found": it looks for key words, so a point made in other words can be missed, and your lecturer reads every written answer). If an answer is still not right after three attempts, the kit also shows a suggested answer with an explanation, so you are never stuck. The answers are **formative**: they are for your learning and for discussion, not for marks. At the end, `kit.submit()` lists anything you have missed and downloads the file so that you can upload it to the VLE.

Each task starts with four short parts: **what you do**, **why it matters**, **a typical example** on a small, different case (a school library), and **what to record**. Read the example before you write code: it shows the same idea on data small enough to follow by eye. `kit.task("T3.3")` shows the same explanation at any time.

Answering again with the same task code replaces your earlier answer and counts as a new attempt, so you can improve an answer at any time. The number of attempts is recorded only so that your lecturer can see which tasks the class found hard; it does not count against you. To try a value without saving it, use `kit.check("T3.3", value)`.

---
## 0 · Setup

**Working in Colab? First make your own copy.** If you opened this notebook from a link your lecturer shared, it is *their* file, and nothing can be saved next to it. Choose **File › Save a copy in Drive**. Colab opens your copy in a new tab (it is kept in **My Drive › Colab Notebooks**, and you can move it to any folder you like); work in that tab from now on. On your own computer (Jupyter or VS Code), just open the notebook from the folder where you want your work to be kept.

Run the next cell once per Colab session. It installs the kit from GitHub (about 20 seconds). If the last line of its output mentions an error, run it again; if it still fails, ask in the VLE forum.

In [ ]:
%pip install -q "ai-learning-kit@https://github.com/charlieabela/ai-learning-kit/archive/refs/tags/v0.5.0.zip"

In [ ]:
import json
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ai_learning_kit as kit

print("ai-learning-kit version:", kit.__version__)

Start your answer book. Replace the name with your own; it is used only to name your answer file.

**Where your work is kept: in the same folder as this notebook.** `kit.start` finds the folder that holds this notebook and keeps everything there: your answer book (`answers/ARI1101_S02_<your name>.json`), the case files (`brightwater_q3/`) and every file you save with a plain name, such as `policy_rules.json`. It prints the full path of this **work folder**.

- On your own computer (Jupyter or VS Code), this is simply the notebook's folder.
- In Colab, the notebook runs on a temporary machine, so `kit.start` connects your **Google Drive** (a window asks you to choose your account and allow access) and finds your notebook there. If your notebook has a name that appears more than once in your Drive, a second window may ask permission to look up which one is open.
- If the notebook is not in your own Drive (you opened your lecturer's shared link, or a GitHub link), `kit.start` says so, keeps your work in **My Drive › ai-learning-kit › ARI1101 › S02** instead, and reminds you to save a copy (see Setup above).

**To choose a different folder**, add `folder=`. In Colab, write the folder as you see it in Google Drive, starting from My Drive, for example `kit.start("ARI1101", "S02", student="Your Name", folder="ARI1101/S02")` for My Drive › ARI1101 › S02 (the folder is created if it does not exist, and Drive is connected for you). On your own computer, a folder name is taken from the notebook's folder.

**Saved, and in Drive?** Colab copies files to Google Drive in the background, so a file you have just saved can take a few minutes to appear when you look at Drive in your browser. Your work is safe in the meantime; `kit.sync()` sends everything immediately, and `kit.submit()` does it for you at the end.

In [ ]:
kit.start("ARI1101", "S02", student="Your Name")

`kit.sources()` lists the evidence, and `kit.workspace()` copies the files into a folder called `brightwater_q3` next to this notebook. From now on you read the files from that folder with your own code, exactly as you would with files a client had sent you.

In [ ]:
display(kit.sources())
folder = kit.workspace()

---
## 1 · Survey the evidence

Before reading any file into Python, look at what it actually contains. `kit.peek(name)` prints the first lines of a file **exactly as stored**, without interpreting them. For binary documents (Word, Excel, PDF) it tells you so, because their bytes are not readable text.

Run the cell and look at each source in turn.

In [ ]:
for name in kit.sources()["source"]:
    print("=" * 30, name, "=" * 30)
    kit.peek(name, lines=8)
    print()

#### Four shapes of data

Data can be organised in four broad **shapes**:

- `table`: rows and columns, every record with the same fields;
- `nested`: objects inside objects, or lists inside records;
- `tree`: elements inside elements, with tags and attributes;
- `document`: running text, possibly with tables and figures.

### Task T1.1 · Format and shape of each source

**What you do**

Use the output of the previous cell (`kit.peek`) and the table from `kit.sources()`. For each of the seven sources, write down two things:

1. its **format**: the kind of file. The file extension tells you (`.csv`, `.json`, `.xml`, `.docx`, `.xlsx`, `.txt`, `.pdf`), and the first characters confirm it.
2. its **shape**: how the information inside is organised, using one of the four words above (`table`, `nested`, `tree`, `document`).

Replace each `___` with a word in quotes, for example `"json"`.

**Why it matters**

The format decides which Python tool can open the file. The shape decides how much work is needed before you can count or filter: a table is ready to use, a nested structure has to be flattened, a tree has to be walked element by element, and a document has to be searched for the pieces you need.

**A typical example** (a different, smaller case: a school library)

A school library keeps three files:

| file | first characters | format | shape | because |
|---|---|---|---|---|
| `loans.csv` | `member_id,book_id,date_out` then `M017,B220,2026-03-02` | `csv` | `table` | a header line, then one line per loan with the same fields |
| `catalogue.json` | `{"books": [{"id": 1, "title": "Dune", "authors": ["Herbert"]}` | `json` | `nested` | a list of books inside an object, and a list of authors inside each book |
| `rules.pdf` | `%PDF-1.7` | `pdf` | `document` | paragraphs written for people to read |

Word (`.docx`) and Excel (`.xlsx`) files both start with the bytes `PK`, because both are ZIP archives. The first bytes cannot tell them apart, so use the file name in `kit.sources()`.

**What to record**

`value`: the completed `survey` dictionary. No reasoning is needed.

In [ ]:
survey = {
    "policy":       {"format": "docx", "shape": "document"},
    "erp":          {"format": "csv",  "shape": "table"},
    "bank":         {"format": ___,    "shape": ___},
    "approvals":    {"format": ___,    "shape": ___},
    "vendors":      {"format": ___,    "shape": ___},
    "emails":       {"format": ___,    "shape": ___},
    "board_report": {"format": ___,    "shape": ___},
}
kit.answer("T1.1", value=survey)

### Task T1.2 · What can be read directly

**What you do**

Two short questions, answered in your own words.

**(a) Which sources can Python read directly, and which need extraction?** Sort the seven sources into two groups:

- *read directly*: the file is already organised data, so one standard call loads it: `pd.read_csv` for CSV, `json.load` for JSON, `ET.parse` (ElementTree) for XML;
- *needs extraction*: the information sits inside a document (Word, PDF, the text of an email), so someone has to find the right words or numbers and turn them into data. In this session the kit does that for you.

The Excel vendor list is a borderline case: `pd.read_excel` can open it, but there are title lines above the table, so you must tell pandas where the header is. Put it in either group, as long as you say why.

**(b) Why can the review question not be answered from any single source?** The question has two parts: *how many payments were above €5,000* and *were they approved as the policy requires*. Say which source gives which part of the answer.

**Why it matters**

Real data science questions rarely come with one tidy file. Knowing which sources are ready to use and which need work is how you estimate the effort, and knowing which source holds which fact is how you plan to combine them.

**A typical example** (a different, smaller case: a school library)

Library audit question: *which members borrowed more books than their limit?*

> `loans.csv` and `members.json` can be read directly, with `pd.read_csv` and `json.load`, because they are already structured. `rules.pdf` needs extraction: the limits are written in a sentence ("students may borrow 5 books"), not stored as data. No single file answers the question: the loans say who borrowed how many books, `members.json` says whether each member is a student or a teacher, and only the rules say what each type may borrow. The three have to be combined.

**What to record**

`reasoning`: three to five sentences covering (a) and (b). No value is needed.

In [ ]:
kit.answer("T1.2", reasoning="""
Write your answer here.
""")

---
## 2 · From a policy document to structured rules

The policy is a Word document written for people. To check transactions against it, its rules have to become **data**. The kit extracts them for you:

In [ ]:
rules = kit.extract.policy_rules()

print(type(rules), "of", len(rules), "items")
print(type(rules[0]))
print(json.dumps(rules[1], indent=2, ensure_ascii=False))

Each rule is a dictionary. Look at rule R2 above: `applies_to_eur` and `approvers` are **dictionaries inside the dictionary**, and `roles` is a **list** inside those. Not every rule has the same keys; print a few more (`rules[3]`, `rules[6]`) to see the differences. If you want to know how the kit found these values in the text, run `kit.explain(kit.extract.policy_rules)`.

### Task T2.1 · Number of policy rules

**What you do**

`kit.extract.policy_rules()` read the Word policy and returned a Python object, stored in the variable `rules`. Find out:

- how many items it holds: `len(rules)`;
- what kind of object it is, and what kind each item is: `type(rules)` and `type(rules[0])`.

The cell above already printed these; read the output.

**Why it matters**

Before you use any data in Python you need to know its type, because the type decides what you can do with it: you loop over a list, look up a key in a dictionary, and filter a DataFrame.

**A typical example** (a different, smaller case: a school library)

```python
shelves = [{"id": "S1", "topic": "History"},
           {"id": "S2", "topic": "Maths"}]
len(shelves)        # 2
type(shelves)       # <class 'list'>
type(shelves[0])    # <class 'dict'>
```
So: *shelves is a list of dictionaries, one per shelf*.

**What to record**

`value`: the number of items. `reasoning`: complete the sentence "rules is a ___ of ___".

In [ ]:
kit.answer("T2.1", value=___, reasoning="rules is a ___ of ___")

### Task T2.2 · Rules round trip through JSON

**What you do**

Save `rules` to a JSON file and read it back:

1. `with open("policy_rules.json", "w", encoding="utf-8") as f:` opens a file for **w**riting and calls it `f`;
2. `json.dump(rules, f, ...)` writes the object into the file;
3. a second `with open(...)` (without `"w"`) opens it for reading, and `json.load(f)` rebuilds the object;
4. `rules_reloaded == rules` compares the two and gives `True` or `False`.

**Why it matters**

Data you extract once should be stored so that it can be reused without extracting it again. A good storage format gives you back exactly what you saved.

**A typical example** (a different, smaller case: a school library)

```python
shelf = {"id": "S1", "books": ["Dune", "Emma"]}

with open("shelf.json", "w", encoding="utf-8") as f:
    json.dump(shelf, f)

with open("shelf.json", encoding="utf-8") as f:
    shelf_again = json.load(f)

shelf_again == shelf      # True: the list inside came back as a list
```

**What to record**

`value`: the result of the comparison, `same`.

In [ ]:
with open("policy_rules.json", "w", encoding="utf-8") as f:
    json.dump(___, f, indent=2, ensure_ascii=False)

with open("policy_rules.json", encoding="utf-8") as f:
    rules_reloaded = ___

same = ___
print("Identical after the round trip:", same)
kit.answer("T2.2", value=same)

### Task T2.3 · Rules round trip through CSV

**What you do**

Now store the same rules as CSV. The cell does the steps for you: it turns `rules` into a DataFrame, writes it to `policy_rules.csv`, reads it back into `rules_csv`, and prints the `approvers` value of rule R2 (row 1) together with its type.

Look closely at that output and answer: what happened to the nested fields (`approvers`, `applies_to_eur`)? Which format would you choose to store the rules, and why?

**Why it matters**

CSV is the most common data format, but it can only hold a grid of text. Knowing what it loses is how you decide when *not* to use it.

**A typical example** (a different, smaller case: a school library)

```python
df = pd.DataFrame([{"id": "S1", "books": ["Dune", "Emma"]}])
df.to_csv("shelf.csv", index=False)
back = pd.read_csv("shelf.csv")

back.loc[0, "books"]          # "['Dune', 'Emma']"   it looks like a list...
type(back.loc[0, "books"])    # <class 'str'>        ...but it is text
back.loc[0, "books"][0]       # "["                  the first character, not "Dune"
```
A CSV cell holds only text, so the list was written as its printed form and came back as a string.

**What to record**

`value`: the reloaded `approvers` value (the cell records it). `reasoning`: what happened to the nested fields, and which format you would use for the rules.

In [ ]:
rules_df = pd.DataFrame(rules)
display(rules_df)

rules_df.to_csv("policy_rules.csv", index=False)
rules_csv = pd.read_csv("policy_rules.csv")

value = rules_csv.loc[1, "approvers"]
print(repr(value), type(value))

kit.answer("T2.3", value=value, reasoning="""
Write your answer here.
""")

### Task T2.4 · Thresholds table as a DataFrame

**What you do**

The policy's table of approval bands comes back exactly as written in Word, so every value is text, including the numbers in `Approvers required`. Convert that column to numbers with `pd.to_numeric` and assign the result back to the same column. Print `thresholds.dtypes` before and after: `object` means text, `int64` means whole numbers.

**Why it matters**

Numbers stored as text look right on screen but cannot be compared or added: `"3" > "10"` is `True` for text. Every time data crosses a format boundary, check the types.

**A typical example** (a different, smaller case: a school library)

```python
t = pd.DataFrame({"band": ["A", "B"], "max_books": ["3", "5"]})
t.dtypes                       # max_books    object   (text)
t["max_books"] = pd.to_numeric(t["max_books"])
t.dtypes                       # max_books    int64
t["max_books"] + 1             # 4, 6
```

**What to record**

`value`: the `thresholds` DataFrame after the conversion.

In [ ]:
thresholds = pd.DataFrame(kit.extract.policy_thresholds())
print(thresholds.dtypes)

thresholds["Approvers required"] = ___
print(thresholds.dtypes)
display(thresholds)

thresholds.to_csv("thresholds.csv", index=False)
kit.answer("T2.4", value=thresholds)

---
## 3 · The ERP payments (CSV)

The finance system's export is a CSV file: one row per payment. Read it yourself with `pd.read_csv` from the `brightwater_q3` folder.

### Task T3.1 · ERP shape

**What you do**

Read the ERP file into a DataFrame called `erp` with `pd.read_csv`. The file is in the case folder: `folder / "erp_payments_q3.csv"` joins the folder and the file name into a path. Then look at the first rows (`erp.head()`), the column types (`erp.dtypes`) and the size (`erp.shape`).

**Why it matters**

The first look at a new table tells you how many records there are, what each column holds and whether pandas guessed the types correctly. Every later step builds on it.

**A typical example** (a different, smaller case: a school library)

```python
loans = pd.read_csv(folder / "loans.csv")
loans.head()      # the first 5 rows
loans.shape       # (250, 3): 250 rows (loans), 3 columns
```
`shape` has no brackets after it: it is a value stored with the DataFrame, not a function you call.

**What to record**

`value`: `erp.shape`, recorded straight after reading, before you add any column.

In [ ]:
erp = pd.read_csv(___)
display(erp.head())
print(erp.dtypes)

kit.answer("T3.1", value=erp.shape)

The histogram below shows how the payment amounts are distributed, with the €5,000 threshold marked. It uses a logarithmic x-axis because the amounts range from a few euro to tens of thousands.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(erp["amount"], bins=[10 ** (x / 10) for x in range(15, 48)], color="#4a6fa5")
ax.set_xscale("log")
ax.axvline(5000, color="#c0504d", ls="--")
ax.text(5300, ax.get_ylim()[1] * 0.85, "€5,000", color="#c0504d")
ax.set_xlabel("amount (log scale)")
ax.set_ylabel("payments")
ax.set_title("ERP payments in Q3 2026")
plt.show()

### Task T3.2 · ERP rows with amount > 5000

**What you do**

Count the rows where `amount` is greater than 5000. A **boolean filter** does this in two steps:

1. `erp["amount"] > 5000` compares every amount with 5000 and gives a column of `True`/`False`;
2. `.sum()` adds them up, counting each `True` as 1.

Ignore the currency for now; that is the next task.

**Why it matters**

Counting the rows that meet a condition is the most common question asked of any table. This first count is also a baseline: the next tasks show how the answer changes when the definition is made precise.

**A typical example** (a different, smaller case: a school library)

```python
books = pd.DataFrame({"title": ["Dune", "Emma", "Ulysses"],
                      "price": [12.0, 30.0, 8.5]})
books["price"] > 10              # True, True, False
(books["price"] > 10).sum()      # 2
```

**What to record**

`value`: the count, a whole number.

In [ ]:
count_any = ___
print(count_any)
kit.answer("T3.2", value=count_any)

### Task T3.3 · ERP payments above €5,000 in euro

**What you do**

Some ERP payments are in US dollars, and rule R7 says the €5,000 threshold applies to the **euro equivalent**. Three steps:

1. **Find the rate in `rules`**, rather than typing it: the R7 dictionary has a key `rate_eur_per_usd`. One way:
   `rate = [r for r in rules if r["rule_id"] == "R7"][0]["rate_eur_per_usd"]`
   (make a list of the rules whose id is R7, take the first, read its rate).
2. **Add a column `amount_eur`**: keep the amount where the currency is EUR, and multiply by the rate otherwise. `.where(condition, other)` keeps a value where the condition is `True` and uses `other` where it is `False`.
3. **Filter** the rows where `amount_eur` is **above** 5000 and keep them in `high`.

**Why it matters**

The policy defines the population to review. Counting with the wrong currency or the wrong comparison (`>=` instead of `>`) gives a different set of payments, and every later step inherits the mistake.

**A typical example** (a different, smaller case: a school library)

```python
fines = pd.DataFrame({"amount": [4.0, 10.0, 8.0],
                      "currency": ["EUR", "GBP", "EUR"]})
rate = 1.17                                   # EUR per GBP
fines["amount_eur"] = fines["amount"].where(fines["currency"] == "EUR",
                                            fines["amount"] * rate)
fines["amount_eur"]                           # 4.0, 11.7, 8.0
over = fines[fines["amount_eur"] > 5]         # the rows above €5
len(over)                                     # 2
```

**What to record**

`value`: `len(high)`, the number of payments above €5,000 in euro.

In [ ]:
rate = ___   # take it from the R7 dictionary in `rules`
erp["amount_eur"] = ___

high = erp[___]
print(len(high), "payments above €5,000 in euro")
kit.answer("T3.3", value=len(high))

### Task T3.4 · Boundary and currency cases

**What you do**

The cell shows two payments that sit on the edge of the definition: one of exactly 5,000.00 euro, and one of 5,300.00 US dollars. For each, decide whether it belongs in the review, and justify your decision by quoting the words of the policy (rules R1 and R7).

**Why it matters**

Edge cases are where definitions matter. A reviewer who cannot say why a borderline payment is in or out cannot defend the count.

**A typical example** (a different, smaller case: a school library)

Library rule: *"members may borrow **more than** 5 books only with the librarian's permission."*

> A member with exactly 5 books needs no permission, because "more than 5" means 6 or more. A member with 6 books does.

**What to record**

`reasoning`: one or two sentences per payment, each quoting the policy.

In [ ]:
display(erp[(erp["amount"] == 5000) | ((erp["amount"] == 5300) & (erp["currency"] == "USD"))])

kit.answer("T3.4", reasoning="""
Write your answer here.
""")

---
## 4 · The bank feed (JSON)

The bank's API returned the statement as one JSON document. Load it with `json.load` and explore its structure before turning anything into a table.

### Task T4.1 · Bank feed top-level keys

**What you do**

Load the bank feed with `json.load(f)` and store it in `bank_doc`. JSON objects become Python dictionaries, so `bank_doc.keys()` lists the names at the top level of the document.

**Why it matters**

A JSON document from an API usually wraps the data you want inside a few layers of information about the data (who sent it, for which period). Looking at the top level first tells you where the records are.

**A typical example** (a different, smaller case: a school library)

```python
with open("catalogue.json", encoding="utf-8") as f:
    catalogue = json.load(f)
type(catalogue)              # <class 'dict'>
list(catalogue.keys())       # ['library', 'updated', 'books']
```
The books are one level down, in `catalogue["books"]`.

**What to record**

`value`: `list(bank_doc.keys())`, the top-level keys of the whole document.

In [ ]:
with open(folder / "bank_feed_q3.json", encoding="utf-8") as f:
    bank_doc = ___

print(type(bank_doc))
print(list(bank_doc.keys()))
print(bank_doc["statement_period"])
kit.answer("T4.1", value=list(bank_doc.keys()))

### Task T4.2 · Bank transactions

**What you do**

The payments are inside the document, under the key `"transactions"`. Take that list out into a variable called `transactions`, count its items with `len`, and print the first item to see how it is nested.

**Why it matters**

Counting the records in each source is the first check that two systems agree. Here the bank and the ERP should describe the same quarter.

**A typical example** (a different, smaller case: a school library)

```python
books = catalogue["books"]   # a list of dictionaries
len(books)                   # 312
books[0]                     # {'id': 1, 'title': 'Dune', 'price': {'value': 12.5, 'currency': 'EUR'}}
```

**What to record**

`value`: `len(transactions)`.

In [ ]:
transactions = ___
print(len(transactions))
print(json.dumps(transactions[0], indent=2))
kit.answer("T4.2", value=len(transactions))

### Task T4.3 · Flattened bank columns

**What you do**

Each transaction holds two objects inside it, `amount` and `counterparty`. `pd.json_normalize(transactions)` turns the list into a DataFrame and **flattens** the inner objects: their keys become separate columns with dotted names such as `amount.value`.

**Why it matters**

pandas can filter and count only columns that hold single values. A column of dictionaries has to be flattened before you can ask "which amounts are above 5000?".

**A typical example** (a different, smaller case: a school library)

```python
books = [{"id": 1, "title": "Dune", "price": {"value": 12.5, "currency": "EUR"}}]
pd.DataFrame(books).columns         # id, title, price          price still holds a dictionary
pd.json_normalize(books).columns    # id, title, price.value, price.currency
```
Dotted columns are read with brackets: `df["price.value"]`, not `df.price.value`.

**What to record**

`value`: `list(bank.columns)`.

In [ ]:
bank = ___
display(bank.head())
kit.answer("T4.3", value=list(bank.columns))

### Task T4.4 · Bank payments above €5,000

**What you do**

Repeat the euro conversion from T3.3 for the bank: use the columns `amount.value` and `amount.currency`, and the same `rate`. Count the transactions above €5,000.

Your count will not match T3.3. Before doing T4.5, suggest in your reasoning what could make a bank statement show more large payments than the finance system.

**Why it matters**

Comparing two independent records of the same events (what finance recorded, and what the bank actually paid) is one of the strongest checks in any audit. A difference is a lead to follow, not an error in your code.

**A typical example** (a different, smaller case: a school library)

The library's till shows 52 fines paid this month, but the fines register lists 50. Possible reasons: a fine was paid twice and both payments went through the till, or a fine was paid but never entered in the register.

**What to record**

`value`: the count. `reasoning`: one or two possible reasons for the difference.

In [ ]:
bank["amount_eur"] = ___
bank_high = bank[___]
print(len(bank_high))

kit.answer("T4.4", value=len(bank_high), reasoning="""
Write your answer here.
""")

### Task T4.5 · Duplicated bank entries

**What you do**

Each bank line has a `remittance_info` text that starts with the ERP reference, for example `TX-2026-01002 Compressor repair`. Two steps:

1. **Extract** the reference into its own column with a regular expression. In `r"(TX-\d{4}-\d{5})"`, `TX-` is matched literally, `\d{4}` means four digits, `\d{5}` means five digits, and the brackets say "this is the part to keep". Lines without a reference get `NaN` (missing).
2. **Find duplicates**: `.duplicated(keep=False)` marks every row whose reference appears more than once. The cell then lists the references.

**Why it matters**

A shared key is what lets you link two sources. Once the bank lines carry the ERP reference, you can see payments that appear twice and payments that finance never recorded.

**A typical example** (a different, smaller case: a school library)

```python
notes = pd.Series(["LN-0001 late return", "LN-0002", "lost card", "LN-0001 reminder"])
refs = notes.str.extract(r"(LN-\d{4})", expand=False)
refs                              # LN-0001, LN-0002, NaN, LN-0001
refs.duplicated(keep=False)       # True, False, False, True
```

**What to record**

`value`: `duplicated_refs`, the sorted list of references that appear more than once.

In [ ]:
bank["erp_ref"] = ___
dupes = bank[bank["erp_ref"].duplicated(keep=False)].sort_values("erp_ref")
display(dupes[["bank_ref", "value_date", "amount.value", "remittance_info"]])

duplicated_refs = sorted(dupes["erp_ref"].unique())
print("No ERP reference:", bank["erp_ref"].isna().sum(), "transaction(s)")
kit.answer("T4.5", value=duplicated_refs)

The duplicates and the payment without an ERP reference are exactly the kind of problem the Scrub session deals with. Today it is enough to have found them: the format (a remittance string inside a nested JSON record) is what made them findable.

---
## 5 · The approval workflow (XML)

The Procurement Workflow System exports one `<Request>` element per approval request. Each request has **attributes** (inside the opening tag) and **child elements**, including a list of `<Approver>` elements. Look at the raw text first with `kit.peek("approvals", lines=14)`.

### Task T5.1 · Number of approval requests

**What you do**

`ET.parse(...).getroot()` reads the XML file and gives you its outermost element, the **root**. `root.findall("Request")` returns a list of the root's direct children whose tag is `Request`. Store it in `requests` and count it. Tags are case-sensitive: `request` would find nothing.

**Why it matters**

In XML, each record is an element. Finding the right elements is the XML equivalent of reading the rows of a table.

**A typical example** (a different, smaller case: a school library)

```python
# library.xml:
# <library>
#   <book id="1" price="12.50"><title>Dune</title></book>
#   <book id="2" price="9.99"><title>Emma</title></book>
# </library>
root = ET.parse("library.xml").getroot()
root.tag                     # 'library'
books = root.findall("book")
len(books)                   # 2
```

**What to record**

`value`: `len(requests)`.

In [ ]:
root = ET.parse(folder / "approvals_pws_q3.xml").getroot()
print(root.tag, root.attrib)

requests = ___     # all <Request> children of the root
print(len(requests))
kit.answer("T5.1", value=len(requests))

### Task T5.2 · Requests as a DataFrame

**What you do**

Turn the XML into a table, one row per request. XML stores values in two places:

- **attributes**, inside the opening tag: `<Request erp_ref="TX-2026-01003" amount="21572.59" ...>`. Read them with `req.get("amount")`.
- **element text**, between the tags: `<Approver ...>Lara Grech</Approver>`. Read it with `.text`.

The loop builds one dictionary per request and adds it to `rows`; `pd.DataFrame(rows)` makes the table. Fill in `amount` and `currency` (attributes) and `n_approvers` (how many `<Approver>` elements: `len(approvers)`). Finally convert `amount` to a number: everything read from XML is text.

**Why it matters**

Building a table from a tree, one record at a time, is the general way to get any XML (or nested JSON) into pandas when a ready-made function does not fit.

**A typical example** (a different, smaller case: a school library)

```python
rows = []
for b in books:                                    # the <book> elements from T5.1's example
    rows.append({"id": b.get("id"),                # attribute
                 "price": b.get("price"),          # attribute, text "12.50"
                 "title": b.find("title").text})   # element text
df = pd.DataFrame(rows)
df["price"] = pd.to_numeric(df["price"])           # now a number
```

**What to record**

`value`: the `pws` DataFrame.

In [ ]:
rows = []
for req in requests:
    approvers = req.findall("Approvers/Approver")
    rows.append({
        "erp_ref": req.get("erp_ref"),
        "amount": ___,
        "currency": ___,
        "n_approvers": ___,
        "roles": "; ".join(a.get("role") for a in approvers),
    })

pws = pd.DataFrame(rows)
pws["amount"] = ___
print(pws.dtypes)
display(pws.head())
kit.answer("T5.2", value=pws)

### Task T5.3 · Requests with fewer than two approvers

**What you do**

Use a boolean filter, as in T3.2, to keep the requests in `pws` with **fewer than two** approvers, and store them in `too_few`. "Fewer than two" means `< 2`.

**Why it matters**

Rule R2 requires two approvers above €5,000. A request with only one is a likely breach, worth flagging before the full check.

**A typical example** (a different, smaller case: a school library)

```python
books[books["n_authors"] < 2]      # books with a single author
```

**What to record**

`value`: `len(too_few)`.

In [ ]:
too_few = ___
display(too_few)
kit.answer("T5.3", value=len(too_few))

### Task T5.4 · Attributes and element text

**What you do**

Look back at the XML and at your loop in T5.2, and at the printout of one `<Approver>` element. Answer three questions:

1. Which values did you read from **attributes**? (In which tag?)
2. Which values are **element text**?
3. Which values needed a **type conversion**, and why?

**Why it matters**

The same information can be put in an attribute or in an element. You have to know which, to write the right code, and you have to remember that XML does not record types.

**A typical example** (a different, smaller case: a school library)

For the library XML: `id` and `price` are attributes of `<book>`, and the title is the element text of `<title>`. `price` had to be converted with `pd.to_numeric` because XML stores everything as text, so `"12.50"` arrives as a string.

**What to record**

`reasoning`: two to four sentences answering the three questions.

In [ ]:
first_approver = requests[0].find("Approvers/Approver")
print(first_approver.attrib, "| text:", first_approver.text)

kit.answer("T5.4", reasoning="""
Write your answer here.
""")

---
## 6 · Combine the evidence and choose a format

You now have three tables: `high` (ERP payments above €5,000), `bank` and `pws`. The review needs one table of high-value payments with their approval information.

### Task T6.1 · Combined high-value table

**What you do**

Combine two tables into one: `high` (ERP payments above €5,000) and `pws` (approval requests). `merge` matches rows that share the same value in a key column, here `erp_ref`.

The `how` argument decides what happens to payments that have **no** matching request:

- `how="inner"` (the default) drops them;
- `how="left"` keeps every row of the left table (`high`) and leaves the request columns empty (`NaN`).

**Why it matters**

Payments without any approval request are exactly the ones the review must find. An inner merge would remove them silently.

**A typical example** (a different, smaller case: a school library)

```python
loans = pd.DataFrame({"book_id": [1, 2, 3]})
books = pd.DataFrame({"book_id": [1, 2], "title": ["Dune", "Emma"]})
loans.merge(books, on="book_id")                # 2 rows: the loan of book 3 is gone
loans.merge(books, on="book_id", how="left")    # 3 rows: book 3 has title NaN
```
Book 3 is the interesting one: a loan of a book that is not in the catalogue.

**What to record**

`value`: the `review` DataFrame.

In [ ]:
review = high.merge(pws[["erp_ref", "n_approvers", "roles"]], on=___, how=___)
review["n_approvers"] = review["n_approvers"].fillna(0).astype(int)

review = review[["erp_ref", "posting_date", "vendor_id", "vendor_name", "amount", "currency",
                 "amount_eur", "n_approvers", "roles"]]
display(review.head())
print(review.shape)
kit.answer("T6.1", value=review)

### Task T6.2 · Choosing a format

**What you do**

The cell saves `review` twice, as `q3_high_value.csv` and as `q3_high_value.json`, compares the file sizes, reads both back and shows the column types side by side. Study the output and answer:

1. Which file is bigger, and why? (Open the JSON with `print(open("q3_high_value.json").read()[:500])` to see.)
2. Which format would you send to the **finance team**, who work in Excel?
3. What does each format lose or keep when the file is read back? Look at the dtypes table.

**Why it matters**

Choosing a format is a decision about the reader: who will open the file, with what tool, and what they need to keep.

**A typical example** (a different, smaller case: a school library)

The library's loans table (250 rows) is 9 KB as CSV and 25 KB as JSON, because JSON repeats the column names in every record. In both, the dates come back as text unless you convert them. For the librarian, who uses Excel, CSV is the natural choice; for the library's website, which reads data through a program, JSON is.

**What to record**

`value`: `sizes` (the cell records it). `reasoning`: your answers to the three questions.

In [ ]:
review.to_csv("q3_high_value.csv", index=False)
review.to_json("q3_high_value.json", orient="records", indent=2)

sizes = {"csv": Path("q3_high_value.csv").stat().st_size,
         "json": Path("q3_high_value.json").stat().st_size}
print(sizes)

back_csv = pd.read_csv("q3_high_value.csv")
back_json = pd.read_json("q3_high_value.json")
display(pd.DataFrame({"original": review.dtypes, "from CSV": back_csv.dtypes, "from JSON": back_json.dtypes}))

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.bar(sizes.keys(), sizes.values(), color=["#4a6fa5", "#e3a33b"])
ax.set_ylabel("bytes")
ax.set_title("The same table in two formats")
plt.show()

kit.answer("T6.2", value=sizes, reasoning="""
Write your answer here.
""")

### Task T6.3 · Extension: convert back to XML

**What you do**

Optional. Write `review` as XML with `review.to_xml(...)` and print the beginning of the file. Compare its structure with the PWS export you read in Part 5: where are the values (attributes or element text)? Is anything nested?

**Why it matters**

Converting a table to a tree format does not create structure that was not there. Seeing that helps you choose formats deliberately.

**A typical example** (a different, smaller case: a school library)

The library loans table written as XML becomes `<data><row><member_id>M017</member_id><book_id>B220</book_id>...</row>...</data>`: one `<row>` per row and one element per column, with no attributes and no nesting.

**What to record**

`reasoning`: two or three sentences. This task is optional.

In [ ]:
review.to_xml("q3_high_value.xml", index=False)
print(Path("q3_high_value.xml").read_text()[:800])

kit.answer("T6.3", reasoning="""
Write your answer here (optional task).
""")

---
## 7 · The black box: checking compliance

Applying eight rules and an exception to every payment is work for the Explore and Interpret sessions. Here the kit does it for you, using **the file you saved**. It reads the ERP references from your CSV and checks each payment against the policy, the approvals, the vendor list and the emails.

In [ ]:
findings = kit.pipeline.compliance_check("q3_high_value.csv")
display(findings[~findings["compliant"]][["erp_ref", "vendor_name", "amount_eur", "band", "issues"]])

If the kit reports that your table is missing transactions, go back to T3.3: the check can only examine what you give it.

### Task T7.1 · Transactions not compliant

**What you do**

`findings` has one row per payment you selected, with a column `compliant` that is `True` or `False`. Count the payments where it is `False`. `~` reverses `True` and `False`, so `(~findings["compliant"]).sum()` counts the `False` values.

**Why it matters**

This number is half of the answer to the review question.

**A typical example** (a different, smaller case: a school library)

```python
checks = pd.DataFrame({"member": ["M1", "M2", "M3"], "ok": [True, False, False]})
(~checks["ok"]).sum()          # 2
(checks["ok"] == False).sum()  # 2, the same count written another way
```

**What to record**

`value`: `n_not_compliant`.

In [ ]:
n_not_compliant = ___
kit.answer("T7.1", value=n_not_compliant)

### Task T7.2 · Split purchases and emails

**What you do**

Rule R4 says that payments to the **same supplier within 7 days** count as a single expenditure. `kit.pipeline.split_purchases()` searches all payments, including those below €5,000, for groups like this. Each row of `splits` is one group, and the column `n_payments` says how many payments it contains.

1. Record how many **payments** are involved in total (add up `n_payments`).
2. Read the Marketing email (`print(kit.extract.emails()[0]["body"])`), then explain: what would a review that only looks at payments above €5,000 miss? What do the emails add that the tables cannot?

**Why it matters**

A question worded as "payments above €5,000" invites a filter on single rows. Splitting a purchase into smaller payments is the classic way around such a rule, and only combining rows, or reading what people wrote, reveals it.

**A typical example** (a different, smaller case: a school library)

The library limit is 5 books. A member borrows 3 books on Monday and 3 on Tuesday: each visit is within the limit, but together they exceed it. An email from the member saying "can you check them out in two goes so it doesn't flag?" shows it was deliberate.

**What to record**

`value`: `n_split_payments`, a number of payments (not of groups). `reasoning`: what the narrow question misses, and what the emails add.

In [ ]:
splits = kit.pipeline.split_purchases()
display(splits)

for m in kit.extract.emails():
    print(m["date"][:10], "|", m["from"], "|", m["subject"])
    print("   references:", m["erp_refs"], "| amounts:", m["amounts_eur"])

Now record your answer to T7.2: the number of payments, and your reasoning.

In [ ]:
n_split_payments = ___
kit.answer("T7.2", value=n_split_payments, reasoning="""
Write your answer here.
""")

---
## 8 · The report

### Task T8.1 · Board report against your count

**What you do**

The kit extracted the Board report's claim: how many large payments it says were made, and how it defines "large" (`board["stated_threshold"]`). Record that number and your own count from T3.3, then explain the difference. Compare the report's wording with the policy's ("above €5,000"), and think about the USD payments.

**Why it matters**

A report is a claim about the data. Checking a claim against the data, and explaining exactly why two numbers differ, is a core skill: it is how you find out whether the difference is a mistake or a different definition.

**A typical example** (a different, smaller case: a school library)

The library's annual report says "40 members borrowed 5 or more books". Your count of members who borrowed *more than* 5 is 37. The 3 members who borrowed exactly 5 explain the difference: the two numbers answer different questions.

**What to record**

`value`: `{"stated": board["stated_high_value_count"], "ours": len(high)}`. `reasoning`: the explanation of the difference.

In [ ]:
board = kit.extract.board_report()
print("Stated:", board["stated_high_value_count"], "payments of", board["stated_threshold"])
display(board["table"])

Now record your answer to T8.1: both counts, and your explanation of the difference.

In [ ]:
kit.answer("T8.1", value={"stated": ___, "ours": ___}, reasoning="""
Write your answer here.
""")

### Task T8.2 · Your findings and the report

**What you do**

Write your findings for the Board in 3 to 6 sentences, in plain language. Cover:

1. the answer to the question: how many payments above €5,000, and how many did not comply;
2. the main kinds of breach (look at the `issues` column of `findings`);
3. one limitation of the review: what it could not see.

The kit then lays out the report (HTML and Word) around your text. In the reasoning, say which format you would send to the Board, and why not CSV or JSON.

**Why it matters**

Analysis is only useful once someone who did not do it can understand and act on it. The format of the final output is chosen for its reader, just like every format before it.

**A typical example** (a different, smaller case: a school library)

> The library audit found that 12 of 250 members borrowed more than their limit in the spring term; 9 of them were students with over 5 books, and 3 were teachers with over 10. Most cases were in the week before exams. The audit used the loan records only, so books taken without being checked out are not included.

**What to record**

`value`: your findings text (`my_findings`). `reasoning`: the format for the Board, and why.

In [ ]:
my_findings = """
Write your findings here. Separate paragraphs with a blank line.
"""

files = kit.report.compliance_report(findings, splits, notes=my_findings, author="Your Name")

kit.answer("T8.2", value=my_findings, reasoning="""
Which format would you send to the Board, and why not CSV or JSON?
""")

---
## Submit

`kit.progress()` shows what you have answered. `kit.submit()` lists anything missing and, in Colab, downloads your answer book. Upload that JSON file to the VLE.

Your answer book is also kept in the `answers` folder next to this notebook (the path that `kit.start` printed), so you can upload it from there instead. If you chose not to connect Google Drive, submit before you close the notebook: Colab deletes its temporary files when it disconnects.

In [ ]:
display(kit.progress())
answer_file = kit.submit()

### Before the next session

Keep this notebook and your answer book. In the Scrub session you will take over the work that the kit did quietly today: the duplicates in the bank feed, the unrecorded payment, the vendor name variants and the currency conversion.